In [1]:
!python -m spacy download es_core_news_sm

C:\Users\alexa\OneDrive\Documentos\GitHub\proyecto_urbania\.venv\Scripts\python.exe: No module named spacy


In [2]:
!pip install --upgrade pip
!pip install numpy pandas
!pip install tf-keras


  Using cached pip-25.3-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.3-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\alexa\OneDrive\Documentos\GitHub\proyecto_urbania\.venv\Scripts\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\alexa\OneDrive\Documentos\GitHub\proyecto_urbania
['.git', '.idea', '.venv', 'Capibara_v4.ipynb', 'ciudad_region.csv', 'data', 'df_limpio.csv', 'Extraccion_1.py', 'inmuebles_maria.csv', 'inmuebles_michael.csv', 'inmuebles_prueba.csv', 'inmuebles_prueba_3.csv', 'inmuebles_prueba_4.csv', 'inmuebles_vf.csv', 'main.py', 'README.md', 'Recomendacion_urbania.ipynb', 'Scrapping']


In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf


BUSQUEDA_CSV = "data/busqueda.csv"
CLIENTES_CSV = "data/clientes.csv"
HISTORIAL_BUSQUEDA = "data/historial_busquedas.csv"

busquedas = pd.read_csv(BUSQUEDA_CSV,
    encoding="latin-1",
    sep=";")
clientes = pd.read_csv(CLIENTES_CSV,    encoding="latin-1",
    sep=";")
historial_busquedas = pd.read_csv(HISTORIAL_BUSQUEDA,    encoding="latin-1",
    sep=";")

busquedas["direccion"] = busquedas["direccion"].fillna("")
busquedas["descripcion"]=busquedas["descripcion"].fillna("")
busquedas["tipo_contrato"] = busquedas["tipo_contrato"].fillna("")
busquedas["tipo_vivienda"] = busquedas["tipo_vivienda"].fillna("")
busquedas["departamento"] = busquedas["departamento"].fillna("")
busquedas["provincia"] = busquedas["provincia"].fillna("")
busquedas["distrito"] = busquedas["distrito"].fillna("")

busquedas["descripcion"] = busquedas["descripcion"] + ' ' + busquedas["tipo_contrato"]+ ' ' + busquedas["tipo_vivienda"]+ ' ' + busquedas["departamento"]+ ' ' + busquedas["provincia"]+ ' ' + busquedas["distrito"]

In [2]:
busquedas.head(5)

,id_busqueda,url,distrito_original,direccion,descripcion,fecha_extraccion,otros,tipo_contrato,tipo_vivienda,precio,area_total,area_cubierta,banos,medios_banos,cochera,dormitorio,antiguedad,departamento,provincia,distrito
0,1,https://urbania.pe/inmueble/clasificado/alclap...,Los Olivos,"LOS CASTAÑOS 1317, 12 De Agosto, Los Olivos",Alquiler Barato Departamento en Los Olivos\nAl...,"45946,83194","LOS CASTAÑOS 1317, 12 De Agosto",alquiler,departamento,1200.0,110.0,110.0,3,0,0,3,5,lima,lima,los olivos
1,2,https://urbania.pe/inmueble/clasificado/alclap...,Ate Vitarte,"De la Unión 303, 27 De Abril, Ate",Minidepartamento de un Dorm. en Tercer Piso / ...,"45946,83194","De la Unión 303, 27 De Abril",alquiler,departamento,1300.0,60.0,60.0,1,0,0,1,5,lima,lima,ate
2,3,https://urbania.pe/inmueble/clasificado/alclap...,Ate Vitarte,"De la Unión 303 , 27 De Abril, Ate","Triplex en Tercer, Cuarto y Quinto Piso / Ate ...","45946,83264","De la Unión 303, 27 De Abril",alquiler,departamento,2650.0,160.0,160.0,1,1,0,3,5,lima,lima,ate
3,4,https://urbania.pe/inmueble/clasificado/alclap...,Miraflores,"a 2 Cuadras del Óvalo de Higuereta, Piso 2, A...","Alquilo Depa, Urb. Alexander Von Humboldt 2do ...","45946,83264","a 2 Cuadras del Óvalo de Higuereta, Piso 2, Al...",alquiler,departamento,3850.0,144.0,144.0,2,0,1,3,20,lima,lima,miraflores
4,5,https://urbania.pe/inmueble/clasificado/alclap...,Cayma,"Cayma Baja, Alto Gloria, Cayma",Departamento Alto de La Gloria - Cayma\nLindo ...,"45946,83264","Cayma Baja, Alto Gloria",alquiler,departamento,2850.0,125.0,110.0,2,1,1,2,8,arequipa,arequipa,cayma


In [3]:
print("Detalle por busqueda:", busquedas.shape)
print("Clientes:", clientes.shape)
print("Historial de busquedas:", historial_busquedas.shape)

print("Columnas busquedas:", busquedas.columns.tolist())
print("Columnas clientes:", clientes.columns.tolist())
print("Columnas historial busqueda:", historial_busquedas.columns.tolist())

Detalle por busqueda: (1673, 20)
Clientes: (4, 6)
Historial de busquedas: (216, 4)
Columnas busquedas: ['id_busqueda', 'url', 'distrito_original', 'direccion', 'descripcion', 'fecha_extraccion', 'otros', 'tipo_contrato', 'tipo_vivienda', 'precio', 'area_total', 'area_cubierta', 'banos', 'medios_banos', 'cochera', 'dormitorio', 'antiguedad', 'departamento', 'provincia', 'distrito']
Columnas clientes: ['id_cliente', 'nombre', 'email', 'ciudad', 'pais', 'fecha_registro']
Columnas historial busqueda: ['id_historial', 'id_cliente', 'id_busqueda', 'fecha_compra']


In [4]:
# Merge historial de busqueda con detalle de busqueda
busqueda_detallada = historial_busquedas.merge(busquedas, on="id_busqueda", how="left")
print("Dataset historial por busqueda unido:", busqueda_detallada.shape)
busqueda_detallada.head()

Dataset historial por busqueda unido: (216, 23)


,id_historial,id_cliente,id_busqueda,fecha_compra,url,distrito_original,direccion,descripcion,fecha_extraccion,otros,...,area_total,area_cubierta,banos,medios_banos,cochera,dormitorio,antiguedad,departamento,provincia,distrito
0,1.0,1.0,142.0,22/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"JR Inca Ripac al 272, Residencial San Felipe,...",Alquiler Minidepartamento Amoblado en Jesus Ma...,"45969,60941","JR Inca Ripac al 272, Residencial San Felipe",...,44.0,44.0,1.0,0.0,1.0,1.0,0.0,lima,lima,jesus maria
1,2.0,1.0,145.0,27/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"Av. Brasil 2477, Jesús María, Residencial San...",Alquiler de Departamento en Jesús María con Pi...,"45969,60965","Av. Brasil 2477, Jesús María, Residencial San ...",...,57.0,57.0,2.0,0.0,0.0,2.0,7.0,lima,lima,jesus maria
2,3.0,1.0,489.0,28/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"Prolongacion quito, Oyague, Jesús María",Alquilo Departamento 3Dorm.\nBeatriz .9.9.2.7....,"45969,67583","Prolongacion quito, Oyague",...,90.0,90.0,2.0,0.0,1.0,3.0,5.0,lima,lima,jesus maria
3,4.0,1.0,876.0,29/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Lima,"AT CDRA 2 DE AV SAN FELIPE, Jesús María, Lima","Jesus Maria 1 Dormitorio 1,200 Soles\nLINDO DE...","45954,0064",AT CDRA 2 DE AV SAN FELIPE,...,135.0,1.0,1.0,0.0,0.0,1.0,1.0,lima,lima,jesus maria
4,5.0,1.0,878.0,4/06/2025,https://urbania.pe/inmueble/clasificado/alclap...,Lima,"Jr lloque yupanqui 1223 , Jesús María, Lima",Mini Departamento Amoblado Wifi Lavadora Terma...,"45954,00676",Jr lloque yupanqui 1223,...,20.0,20.0,1.0,0.0,0.0,1.0,10.0,lima,lima,jesus maria


In [5]:
# Merge con clients
busqueda_full = busqueda_detallada.merge(clientes, on="id_cliente", how="left")
print("Dataset busquedas full:", busqueda_full.shape)
busqueda_full.head()

Dataset busquedas full: (216, 28)


,id_historial,id_cliente,id_busqueda,fecha_compra,url,distrito_original,direccion,descripcion,fecha_extraccion,otros,...,dormitorio,antiguedad,departamento,provincia,distrito,nombre,email,ciudad,pais,fecha_registro
0,1.0,1.0,142.0,22/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"JR Inca Ripac al 272, Residencial San Felipe,...",Alquiler Minidepartamento Amoblado en Jesus Ma...,"45969,60941","JR Inca Ripac al 272, Residencial San Felipe",...,1.0,0.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
1,2.0,1.0,145.0,27/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"Av. Brasil 2477, Jesús María, Residencial San...",Alquiler de Departamento en Jesús María con Pi...,"45969,60965","Av. Brasil 2477, Jesús María, Residencial San ...",...,2.0,7.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
2,3.0,1.0,489.0,28/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"Prolongacion quito, Oyague, Jesús María",Alquilo Departamento 3Dorm.\nBeatriz .9.9.2.7....,"45969,67583","Prolongacion quito, Oyague",...,3.0,5.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
3,4.0,1.0,876.0,29/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Lima,"AT CDRA 2 DE AV SAN FELIPE, Jesús María, Lima","Jesus Maria 1 Dormitorio 1,200 Soles\nLINDO DE...","45954,0064",AT CDRA 2 DE AV SAN FELIPE,...,1.0,1.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
4,5.0,1.0,878.0,4/06/2025,https://urbania.pe/inmueble/clasificado/alclap...,Lima,"Jr lloque yupanqui 1223 , Jesús María, Lima",Mini Departamento Amoblado Wifi Lavadora Terma...,"45954,00676",Jr lloque yupanqui 1223,...,1.0,10.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025


In [6]:
client_profiles = (
    busqueda_full.groupby("id_cliente")
    .apply(lambda g: " ".join(g["direccion"].tolist() + g["descripcion"].tolist()))
    .reset_index(name="texto_cliente")
)

# Agregar información de clientes (ciudad, país)
client_profiles = client_profiles.merge(clientes, on="id_cliente")
client_profiles["texto_cliente"] = (
    client_profiles["nombre"] + " " +
    client_profiles["ciudad"] + " " +
    client_profiles["pais"] + " " +
    client_profiles["texto_cliente"]
)

print("Perfiles de clientes generados:", client_profiles.shape)
client_profiles.head()

Perfiles de clientes generados: (3, 7)


C:\Users\alexa\AppData\Local\Temp\ipykernel_16300\2464208606.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: " ".join(g["direccion"].tolist() + g["descripcion"].tolist()))


,id_cliente,texto_cliente,nombre,email,ciudad,pais,fecha_registro
0,1.0,"Mateo Rojas Cusco Peru JR Inca Ripac al 272, ...",Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
1,2.0,Luis HernÃ¡ndez Lima Peru Calle 30 #300 San Bo...,Luis HernÃ¡ndez,luis.h@email.com,Lima,Peru,15/10/2025
2,3.0,Yury Aquino Lima Peru CALLE PARQUE LAS ORQUIDE...,Yury Aquino,yury300365@hotmail.com,Lima,Peru,20/10/2025


In [7]:
# Remover Stopwords y Lematizar en español en texto_cliente
import spacy
nlp = spacy.load("es_core_news_sm")
print("spaCy cargado: usando lematización y stopwords de spaCy.")

def preprocess_spacy(text):
    doc = nlp(str(text).lower())
    tokens = [t.lemma_ for t in doc if not t.is_stop and t.is_alpha]
    return " ".join(tokens)

# Procesar texto de clientes
client_profiles["texto_cliente_proc"] = client_profiles["texto_cliente"].apply(preprocess_spacy)

# Revisa resultados
client_profiles[["id_cliente", "nombre", "texto_cliente", "texto_cliente_proc"]].head()

spaCy cargado: usando lematización y stopwords de spaCy.


,id_cliente,nombre,texto_cliente,texto_cliente_proc
0,1.0,Mateo Rojas,"Mateo Rojas Cusco Peru JR Inca Ripac al 272, ...",mateo rojo cusco peru jr inco ripac residencia...
1,2.0,Luis HernÃ¡ndez,Luis HernÃ¡ndez Lima Peru Calle 30 #300 San Bo...,luis lima peru calle san borja san borja limo ...
2,3.0,Yury Aquino,Yury Aquino Lima Peru CALLE PARQUE LAS ORQUIDE...,yury aquino lima peru calle parcar orquidea pu...


In [8]:
from sentence_transformers import SentenceTransformer

print("SentenceTransformers disponible. Usando modelo multilingüe.")
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st_model = SentenceTransformer(model_name)

C:\Users\alexa\OneDrive\Documentos\GitHub\proyecto_urbania\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



SentenceTransformers disponible. Usando modelo multilingüe.


In [9]:
import random


# Preprocesar busquedas con spaCy
busquedas["busqueda_direccion"] = busquedas["direccion"].apply(preprocess_spacy)
busquedas["busqueda_descripcion"] = busquedas["descripcion"].apply(preprocess_spacy)

# Listas de textos
direccion_textos = busquedas["busqueda_direccion"].tolist()
descripcion_textos = busquedas["busqueda_descripcion"].tolist()
clientes_textos = client_profiles["texto_cliente_proc"].tolist()

# Todos los textos para embeddings
todos_textos = direccion_textos + descripcion_textos + clientes_textos

# Mostrar 5 elementos aleatorios
muestra_aleatoria = random.sample(todos_textos, 5)

print("Muestra aleatoria de la lista 'todos_textos':")
for i, texto in enumerate(muestra_aleatoria):
    print(f"{i+1}. {texto}")

Muestra aleatoria de la lista 'todos_textos':
1. alquiler almacir industrial arequipar construcción edificio sismo resistenir certifficado respectivo perfecto esado condición operación altura almacén minima metro hombro techo reunir condición suficiente resguardar inclemencia clima sistema drenaje pluvial aéreo traslúcido solicitud cliente piso almacén superficie concreto liso fácil limpieza deber nivel rampa desnivel zona almacenamiento contar cortina rollo metálico m ingreso área arrendado almacén contar iluminación adecuado permitir ejecutar operación centro contar acceso servicio telecomunicación fibra óptico canalización adecuado trabajo recorrido fibra instalación eléctrico canalizada ingreso unidad arrendado almacén contar iluminación adecuado permitir ejecutar operación centro contar acceso servicio telecomunicación fibra óptico canalización adecuado trabajo recorrido fibra instalación eléctrico canalizada ingreso unidad disponibilidad acceso operación almacén hora año patio ma

In [10]:
# Construir textos para ambos grupos
clientes_textos = client_profiles["texto_cliente"].tolist()
busqueda_textos = busquedas["direccion"].tolist()

# Concatenar para un solo encode
todos_textos = clientes_textos + busqueda_textos

# Generar embeddings de todo junto
embeddings = st_model.encode(todos_textos, convert_to_numpy=True, show_progress_bar=True)

# Separar los embeddings en clientes y productos
n_clientes = len(clientes_textos)
n_busquedas = len(busqueda_textos)

print(len(embeddings))
print((n_clientes))
print((n_busquedas))

client_embeddings = embeddings[:n_clientes]
busqueda_embeddings = embeddings[n_clientes:]

Batches: 100%|██████████| 53/53 [00:23<00:00,  2.22it/s]

1676
3
1673


In [12]:
out_dir = "logs/embeddings"
os.makedirs(out_dir, exist_ok=True)

def make_label(text, n_words=5):
    return " ".join(text.split()[:n_words])

# Concatenar embeddings (clientes + productos)
all_embeddings = np.vstack([client_embeddings, busqueda_embeddings])

# Crear metadata
metadata = pd.DataFrame({
    "label": (
        [make_label(t, 5) for t in client_profiles["texto_cliente"].tolist()] +
        [make_label(t, 5) for t in busquedas["direccion"].tolist()]
    ),
    "tipo": (
        ["cliente"] * len(client_profiles) +
        ["direccion"] * len(busquedas)
    ),
    "texto_completo": (
        client_profiles["texto_cliente"].tolist() +
        busquedas["direccion"].tolist()
    )
})

# Guardar metadata como TSV
metadata_path = os.path.join(out_dir, "metadata.tsv")
metadata.to_csv(metadata_path, sep="\t", index=False)

print("Metadata guardada en:", metadata_path)

# Guardar embeddings para TensorBoard
emb_var = tf.Variable(all_embeddings, name="embeddings")
ckpt = tf.train.Checkpoint(embedding=emb_var)
ckpt.save(os.path.join(out_dir, "embedding.ckpt"))

print("Embeddings guardados en:", out_dir)
print("Ejecuta en terminal: tensorboard --logdir logs/embeddings")

Metadata guardada en: logs/embeddings\metadata.tsv
Embeddings guardados en: logs/embeddings
Ejecuta en terminal: tensorboard --logdir logs/embeddings


In [14]:
# Normalizar embeddings de clientes y productos (una vez)
client_embeddings_norm = client_embeddings / np.linalg.norm(client_embeddings, axis=1, keepdims=True)
busqueda_embeddings_norm = busqueda_embeddings / np.linalg.norm(busqueda_embeddings, axis=1, keepdims=True)


def top_k_products_for_client(client_idx, k=3):
    """
    Dado un cliente, devuelve los k busquedas más similares
    que aún NO ha visitado.
    """
    q = client_embeddings_norm[client_idx]  # (dim,)
    sims = np.dot(busqueda_embeddings_norm, q)  # (num_busqueda,)

    # Busquedas ya realizadas por este cliente
    cliente_id = client_profiles.iloc[client_idx]["id_cliente"]
    buscados = set(busqueda_full.loc[busqueda_full["id_cliente"] == cliente_id, "id_busqueda"].tolist())

    # Ordenar por similitud (descendente)
    topk_idx = sims.argsort()[::-1]

    # Filtrar productos que ya compró
    recomendados = []
    for idx in topk_idx:
        pid = busquedas.iloc[idx]["id_busqueda"]
        if pid not in buscados:
            recomendados.append((
                idx,
                sims[idx],
                pid,
                busquedas.iloc[idx]["direccion"],
                busquedas.iloc[idx]["descripcion"]
            ))
        if len(recomendados) >= k:
            break
    return recomendados


# Ejemplo: recomendar sitio al cliente 1 que no haya buscado
cliente_id = client_profiles.iloc[1]["id_cliente"]
print(f"Cliente {cliente_id}: {client_profiles.iloc[1]['texto_cliente'][:120]}...")
print("Top-3 busquedas recomendados (no visitado):")
for idx, score, pid, direccion, desc in top_k_products_for_client(1, k=3):
    print(f" - id_busqueda={pid}, score={score:.4f}, direccion: {direccion}, desc: {desc[:80]}...")

Cliente 2.0: Luis HernÃ¡ndez Lima Peru Calle 30 #300 San Borja,  San Borja, Lima Rousseau 264,  San Borja, Lima San Borja,  San Borja...
Top-3 busquedas recomendados (no visitado):
 - id_busqueda=191, score=0.8350, direccion: SAN Borja Norte 459,  San Borja, Lima, desc: Tranquilidad y Conexión en San Borja Norte: Alquiler Departamento Amoblado, Gas ...
 - id_busqueda=571, score=0.8259, direccion: CALLE DURERO 257, SAN BORJA,  San Borja, Lima, desc: Se Alquila Linda Casa 4 Dorm. - Calle Durero, San Borja S/.7,300
ALQUILO CASA EN...
 - id_busqueda=190, score=0.8189, direccion: San Borja ,  San Borja, Lima, desc: Ic - Departamento en Alquiler San Borja 3 Dorm.
SE ALQUILA DEPARTAMENTO SIN AMOB...


In [15]:
# Ejemplo: recomendar sitio al cliente 1 que no haya buscado
cliente_id = client_profiles.iloc[0]["id_cliente"]
print(f"Cliente {cliente_id}: {client_profiles.iloc[0]['texto_cliente'][:120]}...")
print("Top-3 busquedas recomendados (no visitado):")
for idx, score, pid, direccion, desc in top_k_products_for_client(0, k=3):
    print(f" - id_busqueda={pid}, score={score:.4f}, direccion: {direccion}, desc: {desc[:80]}...")

Cliente 1.0: Mateo Rojas Cusco Peru JR Inca Ripac al 272,  Residencial San Felipe, Jesús María Av. Brasil 2477, Jesús María,  Residen...
Top-3 busquedas recomendados (no visitado):
 - id_busqueda=492, score=0.8108, direccion: Av. San Felipe 601,  Residencial San Felipe, Jesús María, desc: Se Alquila Lindo Departamento Amoblado Jesús Maria
SE ALQUILA LINDO DEPARTAMENTO...
 - id_busqueda=144, score=0.8024, direccion: JR LUIS N SAENZ ,  Residencial San Felipe, Jesús María, desc: Alquilo Acogedor Departamento con Cochera - Jesus Maria
ALQUILO AMPLIO DEPARTAME...
 - id_busqueda=915, score=0.8019, direccion: Av. San Felipe CDRA ,  Jesús María, Lima, desc: Alquiler de Departamento Amoblado en Jesús María - Av San Felipe Cdra 6
Alquiler...


In [16]:
# Ejemplo: recomendar sitio al cliente 1 que no haya buscado
cliente_id = client_profiles.iloc[2]["id_cliente"]
print(f"Cliente {cliente_id}: {client_profiles.iloc[2]['texto_cliente'][:120]}...")
print("Top-3 busquedas recomendados (no visitado):")
for idx, score, pid, direccion, desc in top_k_products_for_client(2, k=3):
    print(f" - id_busqueda={pid}, score={score:.4f}, direccion: {direccion}, desc: {desc[:80]}...")

Cliente 3.0: Yury Aquino Lima Peru CALLE PARQUE LAS ORQUIDEAS 174,  Pueblo Libre, Lima Av Simon Bolivar 947,  San Martin, Pueblo Libr...
Top-3 busquedas recomendados (no visitado):
 - id_busqueda=275, score=0.7992, direccion: Calle Manuel Perez Tudela san isidro,  San Isidro, Lima, desc: Exclusivo Departamento en Alquiler con Vista Al Olivar de San Isidro
Exclusivo y...
 - id_busqueda=342, score=0.7881, direccion: simon bolivar 100 san miguel,  San Miguel, Lima, desc: Alquiler de Comodo Departamento en San Miguel
El edificio se encuentra cerca de ...
 - id_busqueda=338, score=0.7845, direccion: San Miguel,  San Miguel, Lima, desc: Departamento Amoblado en Alquiler San Miguel
Alquiler de Departamento Amoblado y...
